# Benchmark: consultas con y sin índices

**Responsable:** Thomas (lo ejecuta Juan Arias con la base completa)

**Objetivo:** comparar las dos estrategias de almacenamiento de k-mers en tiempo de consulta, con y sin
índices, usando `explain("executionStats")`. De esto depende la conclusión de la decisión 4 de `MODELO.md`.
La carga y el espacio están en `06_benchmark_carga_espacio.ipynb`.

| Estrategia | Colecciones |
| --- | --- |
| Principal (`coleccion`) | `panteon.kmers` (un documento por TE y por k) y `panteon.tes` |
| Alternativa (`embebido`) | `panteon_bench.tes_embebidos` (k-mers en `kmers.k1`…`kmers.k6`) |

Modelo de referencia: `../MODELO.md`

## Instrucciones para ejecutar

1. Revisar los parámetros de la celda siguiente: `URI`, `DB_PRINCIPAL = "panteon"` y `DB_BENCH = "panteon_bench"`
   (las bases restauradas del dump, ver `resultados/dump_info.md`). El resto se deja como está.
2. Cerrar otras aplicaciones (incluido Docker si no es donde corre MongoDB), conectar el cargador y usar
   `caffeinate`, igual que en el benchmark de carga.
3. *Kernel → Restart & Run All*. Tiempo estimado con la base completa: **entre 15 y 30 minutos**; las
   combinaciones más lentas son Q2 y Q4 sobre `tes_embebidos` (unos 7,7 GB sin comprimir). El notebook imprime
   el avance y el tiempo total.
4. El notebook se detiene con un mensaje si falta una colección o un índice, o si las dos estrategias no
   devuelven lo mismo.
5. Subir: este notebook **ejecutado** (con salidas), `resultados/benchmark_consultas.csv` y
   `figuras/benchmark_consultas.png`.

## 0. Parámetros

In [ ]:
URI = "mongodb://localhost:27017"
DB_PRINCIPAL = "panteon"
DB_BENCH = "panteon_bench"
REPETICIONES = 3
N_MUESTRA = 50
SEMILLA = 42
K_UNICO = 2
CSV_SALIDA = "../resultados/benchmark_consultas.csv"
FIGURA = "../figuras/benchmark_consultas.png"

In [ ]:
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pymongo import MongoClient

sys.path.insert(0, "..")
from src.benchmark import guardar_benchmark

K_HEATMAP = 2  # el heatmap del enunciado es de 2-mers

client = MongoClient(URI)
db = client[DB_PRINCIPAL]
db_bench = client[DB_BENCH]
tes, kmers, embebidos = db.tes, db.kmers, db_bench.tes_embebidos

## 1. Verificación del entorno

Colecciones e índices esperados según `MODELO.md` (sección Índices).

In [ ]:
INDICES_ESPERADOS = {
    (DB_PRINCIPAL, "tes"): [
        [("pdb_id", 1)], [("species_id", 1)],
        [("species_name", 1), ("classification.te_superfamily", 1)], [("kingdom", 1)],
    ],
    (DB_PRINCIPAL, "kmers"): [[("te_id", 1), ("k", 1)], [("pdb_id", 1), ("k", 1)], [("k", 1), ("te_order", 1)]],
    (DB_BENCH, "tes_embebidos"): [[("pdb_id", 1)]],
}

print("MongoDB", client.server_info()["version"])
for (base, nombre), esperados in INDICES_ESPERADOS.items():
    assert nombre in client[base].list_collection_names(), \
        f"Falta {base}.{nombre}: restaurar el dump (resultados/dump_info.md)"
    existentes = [[(k, int(v)) for k, v in i["key"].items()] for i in client[base][nombre].list_indexes()]
    faltan = [e for e in esperados if e not in existentes]
    assert not faltan, f"Faltan índices en {base}.{nombre}: {faltan}"
    print(f"{base}.{nombre}: {client[base][nombre].estimated_document_count():,} documentos | "
          f"índices: {len(existentes)}")

n_tes = tes.estimated_document_count()
assert embebidos.estimated_document_count() == n_tes, "tes_embebidos no tiene un documento por TE"
assert kmers.estimated_document_count() == 6 * n_tes, "kmers no tiene 6 documentos por TE"

## 2. Muestra

`N_MUESTRA` identificadores tomados de `tes` con `SEMILLA` fija; la misma muestra para ambas estrategias.

In [ ]:
ids = sorted(tes.distinct("pdb_id"))
assert N_MUESTRA <= len(ids), f"N_MUESTRA = {N_MUESTRA} es mayor que el número de TEs ({len(ids)})"
MUESTRA = random.Random(SEMILLA).sample(ids, N_MUESTRA)
print(len(MUESTRA), "identificadores; primeros:", MUESTRA[:5])

## 3. Consultas

Cada consulta tiene una versión por estrategia. Todas reciben `hint`: `None` (variante `con_indice`, el
plan que elija MongoDB) o `{"$natural": 1}` (variante `sin_indice`, fuerza COLLSCAN sin borrar índices).
Cada función devuelve el resultado ya materializado, así que el tiempo incluye consumir todos los documentos.

- **Q1, perfil de un k:** conteos de `k = K_UNICO` para cada `pdb_id` de la muestra.
- **Q2, heatmap:** frecuencia promedio de cada 2-mer por `te_order`. Frecuencia por TE = `counts / valid_windows`;
  promedio por orden = suma de frecuencias / número de TEs del orden (los k-mers ausentes cuentan como 0;
  no se usa `$avg` después de `$unwind`).
- **Q3, perfil completo:** los 6 perfiles de cada `pdb_id` de la muestra.
- **Q4, sin k-mers:** número de TEs por `kingdom` y `classification.te_order`.

In [ ]:
def opciones(hint):
    return {} if hint is None else {"hint": hint}


# Q1: perfil de un k
def q1_coleccion(hint=None):
    return {p: [(d["valid_windows"], d["counts"]) for d in
                kmers.find({"pdb_id": p, "k": K_UNICO}, {"_id": 0, "valid_windows": 1, "counts": 1}, **opciones(hint))]
            for p in MUESTRA}


def q1_embebido(hint=None):
    campo = f"kmers.k{K_UNICO}"
    return {p: [(d["kmers"][f"k{K_UNICO}"]["valid_windows"], d["kmers"][f"k{K_UNICO}"]["counts"]) for d in
                embebidos.find({"pdb_id": p}, {"_id": 0, campo: 1}, **opciones(hint))]
            for p in MUESTRA}


# Q2: heatmap de 2-mers por orden
def pipeline_q2(filtro, orden, counts, ventanas):
    return [
        {"$match": filtro},
        {"$project": {"_id": 0, "te_order": orden, "f": {"$map": {
            "input": {"$objectToArray": counts},
            "in": {"kmer": "$$this.k", "f": {"$divide": ["$$this.v", ventanas]}},
        }}}},
        {"$facet": {
            "n": [{"$group": {"_id": "$te_order", "n_tes": {"$sum": 1}}}],
            "suma": [{"$unwind": "$f"}, {"$group": {"_id": {"o": "$te_order", "kmer": "$f.kmer"}, "s": {"$sum": "$f.f"}}}],
        }},
        {"$unwind": "$suma"},
        {"$project": {
            "te_order": "$suma._id.o", "kmer": "$suma._id.kmer",
            "frecuencia": {"$divide": ["$suma.s", {"$first": {"$map": {
                "input": {"$filter": {"input": "$n", "cond": {"$eq": ["$$this._id", "$suma._id.o"]}}},
                "in": "$$this.n_tes",
            }}}]},
        }},
    ]


PIPELINE_Q2 = {
    "coleccion": pipeline_q2({"k": K_HEATMAP}, "$te_order", "$counts", "$valid_windows"),
    "embebido": pipeline_q2({}, "$classification.te_order", f"$kmers.k{K_HEATMAP}.counts",
                            f"$kmers.k{K_HEATMAP}.valid_windows"),
}


def q2(coleccion, almacenamiento, hint=None):
    filas = list(coleccion.aggregate(PIPELINE_Q2[almacenamiento], allowDiskUse=True, **opciones(hint)))
    return (pd.DataFrame(filas, columns=["te_order", "kmer", "frecuencia"])
            .pivot(index="te_order", columns="kmer", values="frecuencia").fillna(0.0).sort_index().sort_index(axis=1))


# Q3: perfil completo
def q3_coleccion(hint=None):
    return {p: {d["k"]: (d["valid_windows"], d["counts"]) for d in
                kmers.find({"pdb_id": p}, {"_id": 0, "k": 1, "valid_windows": 1, "counts": 1}, **opciones(hint))}
            for p in MUESTRA}


def q3_embebido(hint=None):
    salida = {}
    for p in MUESTRA:
        d = embebidos.find_one({"pdb_id": p}, {"_id": 0, "kmers": 1}, **opciones(hint))
        salida[p] = {int(k[1:]): (v["valid_windows"], v["counts"]) for k, v in d["kmers"].items()}
    return salida


# Q4: sin k-mers
PIPELINE_Q4 = [{"$group": {"_id": {"kingdom": "$kingdom", "te_order": "$classification.te_order"}, "n": {"$sum": 1}}}]


def q4(coleccion, hint=None):
    return {(d["_id"]["kingdom"], d["_id"]["te_order"]): d["n"]
            for d in coleccion.aggregate(PIPELINE_Q4, allowDiskUse=True, **opciones(hint))}


CONSULTAS = {
    "Q1": {"coleccion": q1_coleccion, "embebido": q1_embebido},
    "Q2": {"coleccion": lambda hint=None: q2(kmers, "coleccion", hint),
           "embebido": lambda hint=None: q2(embebidos, "embebido", hint)},
    "Q3": {"coleccion": q3_coleccion, "embebido": q3_embebido},
    "Q4": {"coleccion": lambda hint=None: q4(tes, hint), "embebido": lambda hint=None: q4(embebidos, hint)},
}
N_BUSQUEDAS = {"Q1": N_MUESTRA, "Q2": 1, "Q3": N_MUESTRA, "Q4": 1}
BASE = {"coleccion": DB_PRINCIPAL, "embebido": DB_BENCH}

## 4. Verificación de correctitud

Antes de medir: cada consulta debe devolver lo mismo en ambas estrategias (Q1 y Q3 para toda la muestra,
Q2 con tolerancia numérica, Q4 exacto). Se comprueba en las dos variantes de índice.

In [ ]:
for hint in [None, {"$natural": 1}]:
    variante = "con_indice" if hint is None else "sin_indice"
    for consulta in ["Q1", "Q3", "Q4"]:
        a, b = CONSULTAS[consulta]["coleccion"](hint), CONSULTAS[consulta]["embebido"](hint)
        if a != b:
            distintos = [p for p in a if a.get(p) != b.get(p)][:5]
            raise AssertionError(f"{consulta} ({variante}) no coincide entre estrategias; ejemplos: {distintos}")
    a, b = CONSULTAS["Q2"]["coleccion"](hint), CONSULTAS["Q2"]["embebido"](hint)
    if not (a.index.equals(b.index) and a.columns.equals(b.columns) and np.allclose(a.values, b.values, rtol=1e-9, atol=1e-12)):
        raise AssertionError(f"Q2 ({variante}) no coincide entre estrategias")
    print(f"{variante}: Q1, Q2, Q3 y Q4 coinciden en ambas estrategias")

assert all(len(v) == 1 for v in CONSULTAS["Q1"]["coleccion"]().values()), "Q1 debe devolver un documento por pdb_id"
assert all(len(v) == 6 for v in CONSULTAS["Q3"]["coleccion"]().values()), "Q3 debe devolver 6 perfiles por pdb_id"
HEATMAP = CONSULTAS["Q2"]["coleccion"]()
print("Heatmap:", HEATMAP.shape[0], "órdenes x", HEATMAP.shape[1], "2-mers")
HEATMAP.round(4)

## 5. `explain("executionStats")`

Una vez por combinación consulta × estrategia × variante. En Q1 y Q3 se explica la búsqueda del primer
`pdb_id` de la muestra. En las agregaciones (Q2 y Q4) las cifras corresponden a la etapa de acceso a la
colección (`$cursor` o el plan empujado al motor de consultas), no a todo el pipeline.

In [ ]:
def buscar(nodo, clave):
    """Primer valor de `clave` en un documento anidado de explain."""
    if isinstance(nodo, dict):
        if clave in nodo:
            return nodo[clave]
        nodo = list(nodo.values())
    if isinstance(nodo, list):
        for v in nodo:
            r = buscar(v, clave)
            if r is not None:
                return r
    return None


def etapas(plan):
    """Etapas del plan ganador, de la raíz a la hoja (p. ej. PROJECTION_SIMPLE>FETCH>IXSCAN)."""
    salida = []
    while isinstance(plan, dict):
        if "stage" in plan:
            salida.append(plan["stage"])
        plan = plan.get("queryPlan") or plan.get("inputStage") or (plan.get("inputStages") or [None])[0]
    return salida


def explicar(base, comando, hint):
    if hint is not None:
        comando = {**comando, "hint": hint}
    salida = client[base].command("explain", comando, verbosity="executionStats")
    stats = buscar(salida, "executionStats") or {}
    plan = etapas(buscar(salida, "winningPlan"))
    return {
        "explain_ms": stats.get("executionTimeMillis"),
        "docs_examinados": stats.get("totalDocsExamined"),
        "claves_examinadas": stats.get("totalKeysExamined"),
        "n_devueltos": stats.get("nReturned"),
        "etapa": plan[-1] if plan else None,
        "plan": ">".join(plan),
    }


p0 = MUESTRA[0]
COMANDOS = {
    ("Q1", "coleccion"): {"find": "kmers", "filter": {"pdb_id": p0, "k": K_UNICO},
                          "projection": {"_id": 0, "valid_windows": 1, "counts": 1}},
    ("Q1", "embebido"): {"find": "tes_embebidos", "filter": {"pdb_id": p0}, "projection": {"_id": 0, f"kmers.k{K_UNICO}": 1}},
    ("Q2", "coleccion"): {"aggregate": "kmers", "pipeline": PIPELINE_Q2["coleccion"], "cursor": {}, "allowDiskUse": True},
    ("Q2", "embebido"): {"aggregate": "tes_embebidos", "pipeline": PIPELINE_Q2["embebido"], "cursor": {}, "allowDiskUse": True},
    ("Q3", "coleccion"): {"find": "kmers", "filter": {"pdb_id": p0}, "projection": {"_id": 0, "k": 1, "valid_windows": 1, "counts": 1}},
    ("Q3", "embebido"): {"find": "tes_embebidos", "filter": {"pdb_id": p0}, "projection": {"_id": 0, "kmers": 1}, "limit": 1},
    ("Q4", "coleccion"): {"aggregate": "tes", "pipeline": PIPELINE_Q4, "cursor": {}, "allowDiskUse": True},
    ("Q4", "embebido"): {"aggregate": "tes_embebidos", "pipeline": PIPELINE_Q4, "cursor": {}, "allowDiskUse": True},
}
VARIANTES = {"con_indice": None, "sin_indice": {"$natural": 1}}

## 6. Medición

Para cada combinación: una ejecución de calentamiento (no se registra) y `REPETICIONES` repeticiones
alternando el orden de las estrategias. Tiempo real con `time.perf_counter()`. En Q1 y Q3 se registra el
tiempo total de las `N_MUESTRA` búsquedas y el promedio por búsqueda. Cada repetición se guarda con
`guardar_benchmark` en `CSV_SALIDA`, con `estrategia = "consulta|almacenamiento|variante"`.

In [ ]:
ALMACENAMIENTOS = ["coleccion", "embebido"]
filas = []
t_inicio = time.perf_counter()

for variante, hint in VARIANTES.items():
    for consulta, funciones in CONSULTAS.items():
        explain = {a: explicar(BASE[a], COMANDOS[(consulta, a)], hint) for a in ALMACENAMIENTOS}
        print(f"[{time.perf_counter() - t_inicio:7.1f} s] {consulta} | {variante}: calentamiento", flush=True)
        for a in ALMACENAMIENTOS:
            funciones[a](hint)
        for repeticion in range(1, REPETICIONES + 1):
            orden = ALMACENAMIENTOS if repeticion % 2 else ALMACENAMIENTOS[::-1]
            nuevas = []
            for a in orden:
                t0 = time.perf_counter()
                funciones[a](hint)
                t = time.perf_counter() - t0
                nuevas.append({
                    "estrategia": f"{consulta}|{a}|{variante}", "base": BASE[a], "repeticion": repeticion,
                    "consulta": consulta, "almacenamiento": a, "variante": variante,
                    "n_busquedas": N_BUSQUEDAS[consulta], "t_total_s": round(t, 6),
                    "t_por_busqueda_ms": round(1000 * t / N_BUSQUEDAS[consulta], 4),
                    **explain[a],
                })
                print(f"[{time.perf_counter() - t_inicio:7.1f} s] {consulta} | {a} | {variante} | "
                      f"repetición {repeticion}: {t:.3f} s", flush=True)
            guardar_benchmark(nuevas, Path(CSV_SALIDA))
            filas += nuevas

t_total = time.perf_counter() - t_inicio
print(f"Tiempo total del benchmark: {t_total / 60:.1f} min")
medidas = pd.DataFrame(filas)

## 7. Resultados

In [ ]:
# Media y desviación estándar muestral (ddof=1) sobre las repeticiones
resumen = (medidas.groupby(["consulta", "almacenamiento", "variante"])
           .agg(repeticiones=("repeticion", "nunique"),
                t_total_media_s=("t_total_s", "mean"), t_total_de_s=("t_total_s", "std"),
                t_busqueda_media_ms=("t_por_busqueda_ms", "mean"), t_busqueda_de_ms=("t_por_busqueda_ms", "std"))
           .round(4))
resumen

In [ ]:
tabla_explain = (medidas.drop_duplicates(["consulta", "almacenamiento", "variante"])
                 .set_index(["consulta", "almacenamiento", "variante"])
                 [["explain_ms", "docs_examinados", "claves_examinadas", "n_devueltos", "etapa", "plan"]])
tabla_explain

In [ ]:
NOMBRES = {
    ("coleccion", "con_indice"): "Colección kmers, con índice",
    ("coleccion", "sin_indice"): "Colección kmers, sin índice",
    ("embebido", "con_indice"): "Embebido, con índice",
    ("embebido", "sin_indice"): "Embebido, sin índice",
}
consultas = sorted(medidas["consulta"].unique())
x = np.arange(len(consultas))
ancho = 0.2

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, ((a, v), nombre) in enumerate(NOMBRES.items()):
    sub = resumen.xs((a, v), level=["almacenamiento", "variante"]).reindex(consultas)
    ax.bar(x + (i - 1.5) * ancho, sub["t_total_media_s"], ancho, yerr=sub["t_total_de_s"].fillna(0),
           capsize=3, label=nombre)
ax.set_yscale("log")
ax.set_xticks(x, [f"{c}\n({N_BUSQUEDAS[c]} búsqueda{'s' if N_BUSQUEDAS[c] > 1 else ''})" for c in consultas])
ax.set_xlabel("Consulta")
ax.set_ylabel("Tiempo total (s, escala log)")
ax.set_title(f"Tiempo de consulta por estrategia ({DB_PRINCIPAL}, media ± DE, n = {REPETICIONES})")
ax.legend(fontsize=8)
fig.tight_layout()
Path(FIGURA).parent.mkdir(exist_ok=True)
fig.savefig(FIGURA, dpi=300)
plt.show()

## 8. Interpretación

<!-- La escribe Thomas con los resultados de la base completa. Discutir:
- Qué estrategia gana en cada consulta (Q1 un k, Q2 heatmap, Q3 perfil completo, Q4 sin k-mers) y por cuánto.
- Efecto de los índices: docs y claves examinados, etapa (IXSCAN frente a COLLSCAN), con y sin índice.
- Relación con la decisión 4 de MODELO.md, junto con la carga y el espacio de 06_benchmark_carga_espacio.ipynb.
-->